<a href="https://colab.research.google.com/github/prasad2154/NewVision/blob/main/GenAI/RAG/Vector_Database_Step_by_Step_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Database - Step-by-Step with Python

### Trainer-Friendly Practical Notebook

This notebook explains **Vector Databases from scratch** using simple Python examples.

### Learning Path

1. Why do we need Vector Databases?
2. What is an Embedding?
3. Convert text into vectors
4. Build a tiny Vector Database manually
5. Understand Cosine Similarity
6. Perform Semantic Search
7. Understand Top-K Retrieval
8. Use a real Vector Database - ChromaDB
9. Store Metadata
10. Filter using Metadata
11. Understand PDF → Chunk → Embedding → Vector DB
12. Build a small retrieval pipeline
13. Understand how Vector DB fits into RAG

---

### Key idea

> **Embedding Model = converts meaning into vectors**  
> **Vector Database = stores and searches those vectors**


1.what is differnce between vectorDB and TraditionalDB

2

# 1. The Problem with Traditional Keyword Search

Consider the following documents.


In [ ]:
documents = [
    "Python is a programming language",
    "Machine learning allows computers to learn from data",
    "Mumbai is the financial capital of India",
    "Cricket is a popular sport in India",
    "Generative AI can create text and images"
]

for i, doc in enumerate(documents, start=1):
    print(f"{i}. {doc}")


1. Python is a programming language
2. Machine learning allows computers to learn from data
3. Mumbai is the financial capital of India
4. Cricket is a popular sport in India
5. Generative AI can create text and images


Suppose the user asks:

> **Which city is important for finance in India?**

The document says:

> **Mumbai is the financial capital of India**

The wording is different, but the **meaning is similar**.

A simple keyword search may struggle when the exact words are different.

This is where **Semantic Search** helps.


# 2. What is an Embedding?

An embedding converts text into a list of numbers called a **vector**.

Conceptually:

```text
"Python programming"
        ↓
Embedding Model
        ↓
[0.12, -0.45, 0.81, 0.33, ...]
```

Texts with similar meanings normally produce vectors that are closer together.

```text
"Python programming" ─────┐
                          ├── Close vectors
"Python coding" ──────────┘


"How to cook biryani" ───── Farther away
```


# 3. Install Required Libraries

Run this cell once.

> If you are using Google Colab or a fresh environment, installation may take a few minutes.


In [ ]:
# Uncomment and run if the packages are not already installed.

!pip install -q sentence-transformers chromadb numpy pandas pypdf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

# 4. Load an Embedding Model

We will use the lightweight model:

`all-MiniLM-L6-v2`

It creates a **384-dimensional vector** for each text input.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


# 5. Convert Documents into Embeddings


In [ ]:
embeddings = model.encode(documents)

print("Number of documents :", len(documents))
print("Embedding shape     :", embeddings.shape)


Number of documents : 5
Embedding shape     : (5, 384)


If the output is:

```text
(5, 384)
```

it means:

- We have **5 documents**
- Each document is represented using **384 numbers**


In [ ]:
print("First document:")
print(documents[0])

print("\nFirst 10 values of its embedding:")
print(embeddings[0][:10])


First document:
Python is a programming language

First 10 values of its embedding:
[-0.03537083  0.038165   -0.04126011  0.01606877 -0.03836703 -0.13447057
  0.03573702  0.04229675 -0.03464679 -0.02643199]


In [ ]:
print("Second document:")
print(documents[1])

print("\nFirst 10 values of its embedding:")
print(embeddings[1][:10])


Second document:
Machine learning allows computers to learn from data

First 10 values of its embedding:
[-0.04632176  0.02776702  0.06456167  0.01702433  0.04806403 -0.02061731
  0.00697891 -0.10290474  0.01618081  0.00837711]


### Important

The individual numbers normally do **not** have a human-readable meaning.

We use the complete vector to compare semantic similarity.


# 6. Build a Tiny Vector Database Manually

Before using ChromaDB, let us understand what a Vector Database conceptually stores.


In [ ]:
vector_database = []

for document, embedding in zip(documents, embeddings):
    vector_database.append({
        "text": document,
        "embedding": embedding
    })

print("Number of records stored:", len(vector_database))


Number of records stored: 5


In [ ]:
print("Text:")
print(vector_database[0]["text"])

print("\nFirst 10 embedding values:")
print(vector_database[0]["embedding"][:10])


Text:
Python is a programming language

First 10 embedding values:
[-0.03537083  0.038165   -0.04126011  0.01606877 -0.03836703 -0.13447057
  0.03573702  0.04229675 -0.03464679 -0.02643199]


Conceptually, each record contains:

```text
ID
Document / Chunk
Vector
Metadata
```

For example:

```text
ID       : 103
Text     : Mumbai is the financial capital of India
Vector   : [0.34, -0.61, 0.18, ...]
Metadata : {"category": "Geography"}
```


# 7. Convert the User Query into a Vector


In [ ]:
query = "Which city is important for finance in India?"

query_embedding = model.encode(query)

print("Query:")
print(query)

print("\nQuery vector shape:")
print(query_embedding.shape)

print("\nFirst 10 values:")
print(query_embedding[:10])


Query:
Which city is important for finance in India?

Query vector shape:
(384,)

First 10 values:
[ 0.06256323  0.01861261 -0.12025727  0.04394081 -0.00257942 -0.01574408
  0.03254154  0.0090136  -0.03429584 -0.03058895]


Notice:

Both the **documents** and the **user query** must pass through the **same embedding model**.

```text
Documents ──→ Embedding Model ──→ Document Vectors

Query ──────→ Same Model ───────→ Query Vector
```


# 8. Understand Cosine Similarity

Now we need to compare:

```text
Query Vector
      VS
Document Vectors
```

One common similarity measure is **Cosine Similarity**.

General interpretation:

- Value closer to **1** → more similar
- Value around **0** → less related
- Negative value → vectors point in largely different directions

The exact meaning of a score depends on the embedding model and data.


In [ ]:
import numpy as np

def cosine_similarity(vector1, vector2):
    return np.dot(vector1, vector2) / (
        np.linalg.norm(vector1) * np.linalg.norm(vector2)
    )


# 9. Compare the Query with Every Document


In [ ]:
for item in vector_database:
    score = cosine_similarity(
        query_embedding,
        item["embedding"]
    )

    print(f"Document : {item['text']}")
    print(f"Score    : {score:.4f}")
    print("-" * 70)


Document : Python is a programming language
Score    : 0.0421
----------------------------------------------------------------------
Document : Machine learning allows computers to learn from data
Score    : 0.1271
----------------------------------------------------------------------
Document : Mumbai is the financial capital of India
Score    : 0.6964
----------------------------------------------------------------------
Document : Cricket is a popular sport in India
Score    : 0.3623
----------------------------------------------------------------------
Document : Generative AI can create text and images
Score    : 0.0429
----------------------------------------------------------------------


The document related to **Mumbai and finance** should receive one of the highest similarity scores.

This is the heart of **Semantic Search**.


# 10. Rank Documents by Similarity


In [ ]:
results = []

for item in vector_database:
    score = cosine_similarity(
        query_embedding,
        item["embedding"]
    )

    results.append({
        "text": item["text"],
        "score": float(score)
    })

results = sorted(
    results,
    key=lambda x: x["score"],
    reverse=True
)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. {result['score']:.4f} | "
        f"{result['text']}"
    )


1. 0.6964 | Mumbai is the financial capital of India
2. 0.3623 | Cricket is a popular sport in India
3. 0.1271 | Machine learning allows computers to learn from data
4. 0.0429 | Generative AI can create text and images
5. 0.0421 | Python is a programming language


# 11. Create a Reusable Semantic Search Function


In [ ]:
def semantic_search(query, documents, model, top_k=3):
    # Step 1: Create embeddings for documents
    document_embeddings = model.encode(documents)

    # Step 2: Create embedding for query
    query_embedding = model.encode(query)

    # Step 3: Compare query with documents
    results = []

    for document, embedding in zip(documents, document_embeddings):
        score = cosine_similarity(
            query_embedding,
            embedding
        )

        results.append({
            "document": document,
            "score": float(score)
        })

    # Step 4: Sort highest similarity first
    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_k]


In [ ]:
query = "Tell me something about artificial intelligence creating content"

results = semantic_search(
    query=query,
    documents=documents,
    model=model,
    top_k=3
)

for result in results:
    print(
        f"{result['score']:.4f} | "
        f"{result['document']}"
    )


0.6063 | Generative AI can create text and images
0.4769 | Machine learning allows computers to learn from data
0.2831 | Python is a programming language


# 12. What is Top-K Retrieval?

Instead of returning only one result, Vector Search often returns the **Top-K most relevant results**.

Example:

```text
K = 3

Query
  ↓
Vector Search
  ↓
Result #1
Result #2
Result #3
```

In RAG applications, values such as **Top-3, Top-5, Top-10** are common starting points, but the best value depends on the application.


# 13. Why Do We Need a Real Vector Database?

Our Python implementation performs:

```python
for item in vector_database:
```

That means every query may compare against every stored vector.

This is acceptable for teaching and small datasets.

But imagine:

```text
1,000 vectors
100,000 vectors
10,000,000 vectors
```

Dedicated vector-search systems use optimized indexes and retrieval algorithms to make similarity search much more efficient.

Examples:

- Chroma
- Pinecone
- Qdrant
- Weaviate
- Milvus

**FAISS** is primarily a vector indexing/search library rather than a complete managed database.


# 14. Create a Real Vector Database using ChromaDB


In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="./vector_db_demo"
)

print("ChromaDB client created.")


ChromaDB client created.


# 15. Create a Collection

A **collection** is a logical place where related vectors/documents are stored.

For teaching purposes, you can loosely compare it to a table, although the internal concepts are different.


In [ ]:
collection = client.get_or_create_collection(
    name="knowledge_base"
)

print("Collection ready.")


Collection ready.


# 16. Add Documents and Embeddings to ChromaDB

We will store:

- ID
- Document
- Embedding
- Metadata


In [ ]:
# Delete old records if you run the notebook repeatedly.
existing = collection.get()

if existing["ids"]:
    collection.delete(ids=existing["ids"])

document_embeddings = model.encode(documents)

collection.add(
    ids=[
        "doc1",
        "doc2",
        "doc3",
        "doc4",
        "doc5"
    ],
    documents=documents,
    embeddings=document_embeddings.tolist()
)

print("Documents stored successfully.")


Documents stored successfully.


# 17. Check Number of Records


In [ ]:
print("Records in collection:", collection.count())


Records in collection: 5


# 18. Search ChromaDB


In [ ]:
query = "Which Indian city is important for financial activity?"

query_embedding = model.encode(query)

results = collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=3
)

print("Question:")
print(query)

print("\nTop Results:")

for i, document in enumerate(results["documents"][0], start=1):
    print(f"{i}. {document}")


Question:
Which Indian city is important for financial activity?

Top Results:
1. Mumbai is the financial capital of India
2. Cricket is a popular sport in India
3. Machine learning allows computers to learn from data


### What happened internally?

```text
User Query
    ↓
Embedding Model
    ↓
Query Vector
    ↓
ChromaDB
    ↓
Similarity Search
    ↓
Top Matching Documents
```


# 19. Inspect Distances Returned by ChromaDB

Depending on the collection configuration, Chroma returns a **distance** value.

Remember:

**Similarity score and distance are not always the same thing.**

For a distance metric, usually:

```text
Smaller distance = closer result
```

Always check the metric used by your vector database.


In [ ]:
results = collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=3
)

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f}")
    print(f"Document: {document}")
    print("-" * 70)


Distance: 0.6566
Document: Mumbai is the financial capital of India
----------------------------------------------------------------------
Distance: 1.2143
Document: Cricket is a popular sport in India
----------------------------------------------------------------------
Distance: 1.8360
Document: Machine learning allows computers to learn from data
----------------------------------------------------------------------


In [ ]:
results

{'ids': [['doc3', 'doc4', 'doc2']],
 'embeddings': None,
 'documents': [['Mumbai is the financial capital of India',
   'Cricket is a popular sport in India',
   'Machine learning allows computers to learn from data']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.6566219329833984, 1.2143425941467285, 1.8360373973846436]]}

# 20. Add Metadata

Real systems usually store metadata along with vectors.

Examples:

- Source filename
- Page number
- Category
- Department
- Document type
- Date
- User ID


In [ ]:
documents_with_metadata = [
    "Python is widely used for Data Science.",
    "Mumbai is the financial capital of India.",
    "Cricket is a popular sport in India.",
    "Article 21 protects life and personal liberty."
]

metadata = [
    {
        "category": "Technology",
        "source": "python_notes.pdf"
    },
    {
        "category": "Geography",
        "source": "india_notes.pdf"
    },
    {
        "category": "Sports",
        "source": "sports_notes.pdf"
    },
    {
        "category": "Law",
        "source": "constitution_notes.pdf"
    }
]

metadata_embeddings = model.encode(
    documents_with_metadata
)


In [ ]:
metadata_collection = client.get_or_create_collection(
    name="metadata_demo"
)

existing = metadata_collection.get()

if existing["ids"]:
    metadata_collection.delete(ids=existing["ids"])

metadata_collection.add(
    ids=[
        "m1",
        "m2",
        "m3",
        "m4"
    ],
    documents=documents_with_metadata,
    embeddings=metadata_embeddings.tolist(),
    metadatas=metadata
)

print("Documents with metadata stored.")


Documents with metadata stored.


# 21. Retrieve Documents with Metadata


In [ ]:
query = "Which article protects personal liberty?"

query_embedding = model.encode(query)

results = metadata_collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=2,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

for document, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("Document :", document)
    print("Metadata :", meta)
    print("Distance :", round(distance, 4))
    print("-" * 70)


Document : Article 21 protects life and personal liberty.
Metadata : {'source': 'constitution_notes.pdf', 'category': 'Law'}
Distance : 0.5469
----------------------------------------------------------------------
Document : Cricket is a popular sport in India.
Metadata : {'source': 'sports_notes.pdf', 'category': 'Sports'}
Distance : 2.08
----------------------------------------------------------------------


# 22. Metadata Filtering

Suppose our database contains documents from many domains.

We can search only inside a particular category.

For example:

```text
Search only Legal documents
```


In [ ]:
query = "which article protects personal liberty"

query_embedding = model.encode(query)

results = metadata_collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=2,
    where={
        "category": "Law"
    }
)

print(results["documents"][0])


['Article 21 protects life and personal liberty.']


Metadata filters are extremely useful in enterprise applications.

Examples:

```text
department = "HR"
user_id = "EMP101"
document_type = "Policy"
year = 2026
case_type = "Criminal"
```

This allows us to combine:

**Semantic Search + Business Rules**


# 23. Vector Database vs Traditional Database

| Traditional Database | Vector Database |
|---|---|
| Exact values | Meaning represented as vectors |
| SQL / exact filters | Similarity search |
| `city = "Mumbai"` | "finance city in India" |
| Good for structured records | Good for semantic/unstructured search |
| IDs, numbers, dates | Embeddings + documents + metadata |

In real applications, they are often used **together**, not as replacements for one another.


# 24. PDF Use Case

Suppose we have a PDF with 100 pages.

We usually do **not** create one embedding for the entire PDF.

Typical pipeline:

```text
PDF
 ↓
Extract Text
 ↓
Split into Chunks
 ↓
Create Embedding for every Chunk
 ↓
Store Chunks + Vectors + Metadata
 ↓
Vector Database
```


# 25. Simple Chunking Example


In [ ]:
text = '''
The Supreme Court of India is the highest judicial authority in India.

Article 32 provides the right to constitutional remedies.

Article 21 protects life and personal liberty.

Article 14 provides equality before law.
'''

chunks = [
    chunk.strip()
    for chunk in text.split("\n\n")
    if chunk.strip()
]

for i, chunk in enumerate(chunks, start=1):
    print(f"Chunk {i}: {chunk}")


# 26. Create Embeddings for Chunks


In [ ]:
chunk_embeddings = model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape :", chunk_embeddings.shape)


# 27. Store Chunks in ChromaDB


In [ ]:
legal_collection = client.get_or_create_collection(
    name="legal_demo"
)

existing = legal_collection.get()

if existing["ids"]:
    legal_collection.delete(ids=existing["ids"])

legal_collection.add(
    ids=[
        f"chunk_{i}"
        for i in range(len(chunks))
    ],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[
        {
            "source": "constitution_demo.txt",
            "chunk_number": i + 1
        }
        for i in range(len(chunks))
    ]
)

print("Legal chunks stored successfully.")


# 28. Ask a Legal Question


In [ ]:
question = "Which article protects personal liberty?"

question_embedding = model.encode(question)

results = legal_collection.query(
    query_embeddings=[
        question_embedding.tolist()
    ],
    n_results=2,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

print("Question:")
print(question)

print("\nRetrieved Context:")

for document, meta in zip(
    results["documents"][0],
    results["metadatas"][0]
):
    print()
    print(document)
    print("Source:", meta)


The Vector Database did **not answer the question**.

It performed:

```text
SEARCH / RETRIEVAL
```

This distinction is very important.

The retrieved content can later be passed to an **LLM** for answer generation.


# 29. A Simple Retrieval Function


In [ ]:
def retrieve_relevant_chunks(
    question,
    collection,
    model,
    top_k=3
):
    question_embedding = model.encode(question)

    results = collection.query(
        query_embeddings=[
            question_embedding.tolist()
        ],
        n_results=top_k
    )

    return results["documents"][0]


In [ ]:
question = "What protects equality before law?"

retrieved_chunks = retrieve_relevant_chunks(
    question=question,
    collection=legal_collection,
    model=model,
    top_k=2
)

for i, chunk in enumerate(
    retrieved_chunks,
    start=1
):
    print(f"{i}. {chunk}")


# 30. Build the Context for an LLM

A RAG application typically joins the retrieved chunks into a context.


In [ ]:
context = "\n\n".join(retrieved_chunks)

print("CONTEXT FOR LLM:")
print("=" * 70)
print(context)


We can then create a prompt such as:


In [ ]:
prompt = f'''
Use only the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
'''

print(prompt)


# 31. Where Does the Vector Database Fit in RAG?

```text
                    RAG PIPELINE

Documents / PDFs
      ↓
Text Extraction
      ↓
Chunking
      ↓
Embedding Model
      ↓
Vector Database
      ↓
-----------------------------
      ↓
User Question
      ↓
Embedding Model
      ↓
Query Vector
      ↓
Vector Search
      ↓
Top-K Relevant Chunks
      ↓
LLM + Retrieved Context
      ↓
Final Answer
```

### Important

The **Vector Database retrieves information**.

The **LLM generates the final natural-language answer**.


# 32. Optional: Read a Real PDF

Place a PDF file in the same folder as the notebook and change the filename below.

This cell uses `pypdf`.


In [ ]:
# OPTIONAL CELL

# from pypdf import PdfReader

# pdf_path = "sample.pdf"

# reader = PdfReader(pdf_path)

# pdf_text = ""

# for page_number, page in enumerate(reader.pages, start=1):
#     page_text = page.extract_text() or ""
#     pdf_text += page_text + "\n"

# print("Characters extracted:", len(pdf_text))
# print(pdf_text[:2000])


# 33. Simple Character-Based Chunking for PDF Text

For production applications, libraries such as LangChain or LlamaIndex provide more advanced splitters.

Here we use a simple function so that the concept remains clear.


In [ ]:
def simple_chunk_text(
    text,
    chunk_size=500,
    overlap=100
):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start = end - overlap

    return chunks


In [ ]:
demo_text = (
    "Vector databases are useful for semantic search. "
    "Embeddings represent text as vectors. "
    "Similar meanings produce nearby vectors. "
) * 20

demo_chunks = simple_chunk_text(
    demo_text,
    chunk_size=250,
    overlap=50
)

print("Number of chunks:", len(demo_chunks))

for i, chunk in enumerate(
    demo_chunks[:3],
    start=1
):
    print()
    print(f"Chunk {i}")
    print(chunk)


# 34. Why Do We Use Chunk Overlap?

Suppose one important sentence lies at the boundary between two chunks.

Without overlap:

```text
Chunk 1 ends here → important idea gets cut
Chunk 2 starts here
```

With overlap:

```text
Chunk 1 ───────────────
             ─────────────── Chunk 2
             ↑ shared text
```

Overlap helps preserve some context across chunk boundaries.

But excessive overlap also increases:

- Storage
- Embedding cost
- Duplicate retrieval

So it should be tuned.


# 35. Classroom Exercise 1

Add these documents:

```text
Delhi is the capital of India.
Taj Mahal is located in Agra.
Bengaluru is known as India's Silicon Valley.
Sachin Tendulkar is a famous Indian cricketer.
Python is widely used in Artificial Intelligence.
```

Then ask:

```text
Which Indian city is famous for technology companies?
```

### Task

1. Create embeddings
2. Store vectors
3. Create query embedding
4. Calculate similarities
5. Return Top-3 results


In [ ]:
# Write your solution here


# 36. Classroom Exercise 2

Using ChromaDB, store information from these categories:

- Technology
- Sports
- Geography
- Law

Add appropriate metadata.

Then search:

```text
What protects personal freedom?
```

Filter the query so that only:

```text
category = Law
```

is searched.


In [ ]:
# Write your solution here


# 37. Important Terminology

### Vector

A list of numbers representing an item.

```text
[0.12, -0.52, 0.81, ...]
```

### Embedding

A vector representation produced by an embedding model.

### Embedding Model

Model that converts text/images/etc. into vector representations.

### Vector Database

System optimized for storing and searching vectors.

### Similarity Search

Finding vectors close to the query vector.

### Semantic Search

Searching by **meaning**, not just exact keywords.

### Top-K

Number of most relevant results returned.

### Metadata

Additional information associated with each vector.

### Chunk

Smaller section of a larger document.

### Retrieval

Finding relevant information from the Vector Database.

### RAG

Retrieval-Augmented Generation:

```text
Retrieve relevant context
          +
Generate an answer using an LLM
```


# 38. Final Comparison

### Traditional Search

```text
Query:
financial capital India

        ↓

Keyword Matching

        ↓

Documents containing similar words
```

### Vector Search

```text
Query:
Which Indian city is important for finance?

        ↓

Embedding Model

        ↓

Query Vector

        ↓

Vector Similarity

        ↓

Mumbai is the financial capital of India
```


# 39. The Most Important Diagram

```text
                INDEXING PHASE

Documents / PDFs
       │
       ▼
    Chunking
       │
       ▼
Embedding Model
       │
       ▼
     Vectors
       │
       ▼
┌─────────────────────┐
│   VECTOR DATABASE   │
│                     │
│ ID                  │
│ Vector              │
│ Document / Chunk    │
│ Metadata            │
└─────────────────────┘


                 SEARCH PHASE

User Question
       │
       ▼
Embedding Model
       │
       ▼
  Query Vector
       │
       ▼
 Vector Database
       │
       ▼
Similarity Search
       │
       ▼
Top-K Relevant Chunks
       │
       ▼
      LLM
       │
       ▼
 Final Answer
```


# 40. Final Takeaway

Remember these four roles:

| Component | Main Responsibility |
|---|---|
| Document / Chunk | Contains the original knowledge |
| Embedding Model | Converts meaning into vectors |
| Vector Database | Stores and retrieves similar vectors |
| LLM | Generates the final answer |

### One-line definition

> **A Vector Database stores vector representations of data and efficiently retrieves items that are semantically similar to a query.**

### Complete flow

```text
Data
 ↓
Chunking
 ↓
Embedding
 ↓
Vector DB
 ↓
Query Embedding
 ↓
Similarity Search
 ↓
Top-K Context
 ↓
LLM
 ↓
Answer
```
